In [ ]:
import mdtraj as md
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted

In [ ]:
pdb_path = natsorted(list(Path('../../analysis/met_samples/model-532-all-macrostates-1000-samples/').glob('processed*')))
pdb_path

In [ ]:
states = {i : md.load(str(path/'merged.pdb')) for i, path in enumerate(pdb_path)}

In [ ]:
aloop_res = states[0].topology.select('resid 155 to 180')
states_subset = {i: traj.atom_slice(aloop_res) for i, traj in states.items()}

In [ ]:
labels = ["PHE1223", "GLY1224", "LEU1225", "ALA1226", "ARG1227", "ASP1228", "MET1229", "TYR1230", "ASP1231", "LYS1232", "GLU1233", "TYR1234", "TYR1235", "SER1236", "VAL1237", "HIS1238", "ASN1239", "LYS1240", "THR1241", "GLY1242", "ALA1243", "LYS1244", "LEU1245", "PRO1246", "VAL1247", "LYS1248"]

In [ ]:
def plot_dssp_stacked(states, fname):
    # Define DSSP categories and colors
    dssp_codes = ['H', 'B', 'E', 'G', 'I', 'T', 'S', ' ']
    dssp_labels = ['Alpha helix', 'Beta bridge', 'Beta strand', '3_10 helix', 'Pi helix', 
                   'Turn', 'Bend', 'Loop']
    dssp_colors = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'brown', 'gray']
    
    # Compute residue-wise counts for each DSSP category
    state_dssp = {k:md.compute_dssp(v, simplified=False) for k,v in states.items()}
    state_dssp_counts = {}
    for state, dssp_matrix in state_dssp.items():
        n_residues = dssp_matrix.shape[1]  # Number of residues
        counts = np.zeros((len(dssp_codes), n_residues))  # Matrix of counts (DSSP x Residue)
        
        for i, code in enumerate(dssp_codes):
            counts[i, :] = np.sum(dssp_matrix == code, axis=0)  # Count occurrences
        
        state_dssp_counts[state] = counts / dssp_matrix.shape[0]  # Normalize to fraction
    
    # Generate stacked bar plots
    fig, axes = plt.subplots(len(state_dssp_counts), 1, figsize=(10, 2.4 * len(state_dssp_counts)), sharex=True)
    
    if len(state_dssp_counts) == 1:
        axes = [axes]  # Ensure iterable
    
    for ax, (state, counts) in zip(axes, state_dssp_counts.items()):
        bottom = np.zeros(counts.shape[1])  # Initialize bottom for stacking
        for i, (label, color) in enumerate(zip(dssp_labels, dssp_colors)):
            ax.bar(np.arange(counts.shape[1]), counts[i], bottom=bottom, color=color, label=label)
            bottom += counts[i]  # Update bottom for stacking
        ax.set_ylim([0,1])
        ax.set_ylabel(f"State {state+1}", fontsize=14)
        
    axes[0].legend(loc="upper right", fontsize=10)
    plt.xlim([-0.5, counts.shape[1] - 0.5])
    axes[-1].set_xticks(np.arange(0, len(labels)))
    axes[-1].set_xticklabels(labels, rotation=60, fontsize=12)
    plt.tight_layout()
    if fname is not None:
        plt.savefig(fname)
    plt.show()

In [ ]:
plot_dssp_stacked(states_subset, 'mdtraj_dssp.pdf')